# Hanson Hoops preview research

Cache-only examples for the five-season regular-season window. Acquisition/import is deliberately separate from this notebook. Missing caches raise an explicit error rather than returning zeros.

In [ ]:
from preview_research.notebook import (
    DEFAULT_SEASONS, ResearchStore, compare_entity, export_result,
    filter_rows, join_compatible, load_dataset, year_over_year,
)
store = ResearchStore()
DEFAULT_SEASONS

## Coverage and validation state

In [ ]:
store.coverage().sort_values(['season', 'dataset'])

## One player across two actual seasons

In [ ]:
shooting = load_dataset('player_shooting', ['2024-25', '2025-26'])
compare_entity(
    shooting, 'Shai Gilgeous-Alexander',
    metrics=['PTS', 'FG3A', 'FG3_PCT', 'CALC_EFG_PCT', 'CALC_TS_PCT'],
).frame

## Three-point percentage increases with an attempt threshold

In [ ]:
changes = year_over_year(shooting, 'CALC_FG3_PCT')
improved = filter_rows(
    changes,
    conditions=['CHANGE_CALC_FG3_PCT>0', 'FG3A>=200'],
    sort_by='CHANGE_CALC_FG3_PCT', descending=True, limit=25,
    columns=['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'SEASON', 'FG3M', 'FG3A', 'CALC_FG3_PCT', 'CHANGE_CALC_FG3_PCT'],
)
improved.frame

In [ ]:
export_result(improved, 'notebook-three-point-improvers.csv', notes='Attempt threshold applies to the later-season row.')

## Join totals to per-100

Use total-season `MIN` from Base Totals for eligibility; `MIN` in a Per100 table is not total minutes.

In [ ]:
totals = load_dataset('player_base_totals', ['2025-26'])
per100 = load_dataset('player_per100', ['2025-26'])
joined = join_compatible(totals, per100, keys=['PLAYER_ID', 'SEASON', 'SEASON_TYPE'])
leaders = filter_rows(
    joined, conditions=['MIN_LEFT>=1000'], sort_by='PTS_RIGHT', descending=True, limit=20,
    columns=['PLAYER_ID', 'PLAYER_NAME_LEFT', 'TEAM_ABBREVIATION_LEFT', 'GP_LEFT', 'MIN_LEFT', 'PTS_RIGHT'],
)
leaders.frame

## Team ranking after team tables are acquired

Uncomment after `team_advanced` exists for both seasons.

In [ ]:
# teams = load_dataset('team_advanced', ['2024-25', '2025-26'])
# team_changes = year_over_year(teams, 'NET_RATING')
# filter_rows(team_changes, conditions=['CHANGE_NET_RATING>-100'], sort_by='CHANGE_NET_RATING', descending=True).frame

## One team's player on/off table

On/off is descriptive, not causal impact and not Pair Fit. Uncomment after acquiring this exact team-season.

In [ ]:
# on_off = load_dataset('player_on_off', ['2025-26'], team_id='1610612760')
# okc = filter_rows(on_off, sort_by='MIN', descending=True, columns=['TEAM_ID', 'TEAM_NAME', 'VS_PLAYER_ID', 'VS_PLAYER_NAME', 'ON_OFF', 'GP', 'MIN', 'OFF_RATING', 'DEF_RATING', 'NET_RATING'])
# export_result(okc, '2025-26-okc-player-on-off.csv', notes='Descriptive split; not causal impact.')